# TVAE

sdv docu: https://docs.sdv.dev/sdv  
sdv-quickstart: https://colab.research.google.com/drive/1F3WWduNjcX4oKck6XkjlwZ9zIsWlTGEM?usp=sharing#scrollTo=JfSdHdDnjmUO

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/raw/yeast.csv")

data = df[(df["localization_site"] == "CYT") | (df["localization_site"] == "ME3")]
data["localization_site"] = data["localization_site"].map({"CYT":0, "ME3": 1})

/var/folders/gv/d21k9b7d7v9d382g4l3v4rqm0000gn/T/ipykernel_72745/3547207569.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["localization_site"] = data["localization_site"].map({"CYT":0, "ME3": 1})


In [3]:
data["localization_site"].value_counts()

localization_site
0    463
1    163
Name: count, dtype: int64

In [4]:
data

,mcg,gvh,alm,mit,erl,pox,vac,nuc,localization_site
5,0.51,0.40,0.56,0.17,0.5,0.5,0.49,0.22,0
9,0.40,0.39,0.60,0.15,0.5,0.0,0.58,0.30,0
12,0.40,0.42,0.57,0.35,0.5,0.0,0.53,0.25,0
15,0.46,0.44,0.52,0.11,0.5,0.0,0.50,0.22,0
16,0.47,0.39,0.50,0.11,0.5,0.0,0.49,0.40,0
...,...,...,...,...,...,...,...,...,...
1475,0.71,0.50,0.50,0.18,0.5,0.0,0.46,0.22,0
1476,0.61,0.48,0.54,0.25,0.5,0.0,0.50,0.22,0
1477,0.38,0.32,0.64,0.41,0.5,0.0,0.44,0.11,0
1478,0.38,0.40,0.66,0.35,0.5,0.0,0.43,0.11,0


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 626 entries, 5 to 1483
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   mcg                626 non-null    float64
 1   gvh                626 non-null    float64
 2   alm                626 non-null    float64
 3   mit                626 non-null    float64
 4   erl                626 non-null    float64
 5   pox                626 non-null    float64
 6   vac                626 non-null    float64
 7   nuc                626 non-null    float64
 8   localization_site  626 non-null    int64  
dtypes: float64(8), int64(1)
memory usage: 48.9 KB


In [6]:
data_min = data[data["localization_site"] == 1]

In [7]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import Metadata

In [8]:
metadata = Metadata.detect_from_dataframe(
    data=data_min,
    table_name='yeast')

In [9]:
metadata

{
    "tables": {
        "yeast": {
            "columns": {
                "mcg": {
                    "sdtype": "numerical"
                },
                "gvh": {
                    "sdtype": "numerical"
                },
                "alm": {
                    "sdtype": "numerical"
                },
                "mit": {
                    "sdtype": "numerical"
                },
                "erl": {
                    "sdtype": "numerical"
                },
                "pox": {
                    "sdtype": "categorical"
                },
                "vac": {
                    "sdtype": "numerical"
                },
                "nuc": {
                    "sdtype": "numerical"
                },
                "localization_site": {
                    "sdtype": "categorical"
                }
            }
        }
    },
    "relationships": [],
    "METADATA_SPEC_VERSION": "V1"
}

In [11]:
metadata.save_to_json("metadata_yeast_v1.json")

In [10]:
synthesizer = TVAESynthesizer(metadata) 

/Users/markusmuller/micromamba/envs/sdg-env/lib/python3.11/site-packages/sdv/single_table/base.py:105: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [11]:
synthesizer.get_parameters()

{'enforce_min_max_values': True,
 'enforce_rounding': True,
 'embedding_dim': 128,
 'compress_dims': (128, 128),
 'decompress_dims': (128, 128),
 'l2scale': 1e-05,
 'batch_size': 500,
 'verbose': False,
 'epochs': 300,
 'loss_factor': 2,
 'cuda': True}

In [12]:
synthesizer.fit(data_min)

In [13]:
# asses the loass values
synthesizer.get_loss_values()

,Epoch,Batch,Loss
0,0,0,18.878059
1,1,0,17.131699
2,2,0,15.215642
3,3,0,14.337144
4,4,0,14.343128
...,...,...,...
295,295,0,-14.903847
296,296,0,-14.708637
297,297,0,-14.962408
298,298,0,-14.236108


In [ ]:
# code to save the synthesizer
#synthesizer.save(filepath="path.pkl")

# code to load the synthesizer
# synthesizer = CTGANSynthesizer.load(filepath="path.pkl")

In [15]:
syn_data = synthesizer.sample(num_rows=10)

In [16]:
syn_data

,mcg,gvh,alm,mit,erl,pox,vac,nuc,localization_site
0,0.33,0.47,0.37,0.21,0.5,0.0,0.51,0.22,1
1,0.50,0.70,0.38,0.17,0.5,0.0,0.52,0.24,1
2,0.38,0.48,0.42,0.12,0.5,0.0,0.51,0.23,1
3,0.36,0.44,0.39,0.14,0.5,0.0,0.51,0.22,1
4,0.32,0.48,0.24,0.13,0.5,0.0,0.54,0.23,1
5,0.50,0.48,0.38,0.20,0.5,0.0,0.54,0.22,1
6,0.40,0.48,0.44,0.17,0.5,0.0,0.52,0.23,1
7,0.44,0.47,0.42,0.19,0.5,0.0,0.49,0.22,1
8,0.50,0.43,0.38,0.13,0.5,0.0,0.57,0.23,1
9,0.37,0.40,0.36,0.15,0.5,0.0,0.54,0.22,1


In [17]:
# Evaluate real vs. synthetic data
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality

In [18]:
# https://docs.sdv.dev/sdv/single-table-data/evaluation/diagnostic

diagnostic = run_diagnostic(
    real_data=data_min,
    synthetic_data=syn_data,
    metadata=metadata
)#

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 9/9 [00:00<00:00, 360.87it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 651.09it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [19]:
# https://docs.sdv.dev/sdv/single-table-data/evaluation/data-quality

quality_report = evaluate_quality(
    real_data=data_min,
    synthetic_data=syn_data,
    metadata=metadata)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1013.06it/s]|
Column Shapes Score: 76.95%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 440.92it/s]|
Column Pair Trends Score: 62.27%

Overall Score (Average): 69.61%

